# RKS f1mo 分解 (TPSS0, MGGA)<br>DFT xc 贡献的 f1ao / f1mo 实现

本文档对标 `02-4-decomp_cphf_1.ipynb` (RHF 的 f1ao / f1mo 分解)，但着重于 **RKS 特有的 DFT 格点积分部分**。

**f1ao** (PySCF 中称为 `h1` 或 `make_h1` 的返回值) 是 CP-KS 方程的右端项，形状为 `(natm, 3, nao, nao)`，代表一阶 Fock 矩阵的 skeleton 导数。

对于 RKS，其成分为：
$$
\mathrm{f1ao} = \mathrm{h1ao} + \mathrm{J1ao} - \frac{c_K}{2} \mathrm{K1ao} + \mathrm{vxc\_deriv1}
$$

其中 $\mathrm{h1ao}$ (核 Hamilton)、$\mathrm{J1ao}$ (Coulomb)、$\mathrm{K1ao}$ (交换) 都与 RHF 完全相同。**$\mathrm{vxc\_deriv1}$ 是新增的 DFT xc skeleton 导数**。

本文档将：
1. 使用 PySCF 库函数获取 f1ao 参考值与其各部分
2. 从原理出发，自行实现 $\mathrm{vxc\_deriv1}$ (即 `_get_vxc_deriv1`)
3. 变换到 MO 基得到 f1mo → `(natm, 3, nmo, nocc)`

其中 $\mathrm{vxc\_deriv1}$ 包含两部分：
- **"ipip" 部分**：梯度级别的 Vxc 矩阵 (与 `pyscf.grad.rks.get_vxc` 相同)
- **"fxc" 部分**：密度在格点上的变化 $\partial_{A_t} \xi_g^\chi$ 通过 fxc 核 $f^{\chi\chi'}$ 的反馈

In [1]:
from pyscf import gto, dft, lib
from pyscf.hessian import rks as rks_hess
from pyscf.df.hessian import rhf as df_rhf_hess
from pyscf.grad import rks as rks_grad
from pyscf.dft import numint
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = dft.RKS(mol, xc="TPSS0").density_fit()
dat0 = np.load("nh3_r_tpss0.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
aoslices = mol.aoslice_by_atom()
ni = mf._numint

## 1. Reference f1ao from PySCF

In [5]:
# PySCF 的 make_h1 返回 f1ao (CPHF/CPKS 的右端)
mf_hess = mf.Hessian()
mf_hess.auxbasis_response = 2
f1ao_ref = mf_hess.make_h1(mo_coeff, mo_occ)
print("f1ao_ref shape:", f1ao_ref.shape)
print("f1ao_ref fp:   ", lib.fp(f1ao_ref))


WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.



f1ao_ref shape: (4, 3, 49, 49)
f1ao_ref fp:    -2.4170164424421454


## 2. f1ao 分解: hcore, J, K, Vxc_deriv1

In [6]:
# Hybrid 系数
omega, alpha, hyb = ni.rsh_and_hybrid_coeff(mf.xc, spin=mol.spin)
print(f"omega={omega}, hyb={hyb}")

# Vxc_deriv1 (DFT xc 部分)
vxc_deriv1_ref = rks_hess._get_vxc_deriv1(mf_hess, mo_coeff, mo_occ, 4000)
print("vxc_deriv1_ref fp:", lib.fp(vxc_deriv1_ref))

# J/K 部分 (from DF-RHF _gen_jk)
gen_jk = list(df_rhf_hess._gen_jk(mf_hess, mo_coeff, mo_occ, with_k=True))
h1ao = np.array([r[1] for r in gen_jk])  # hcore derivative
j1ao = np.array([r[2] for r in gen_jk])  # J derivative
k1ao = np.array([r[3] for r in gen_jk])  # K derivative
print("h1ao (hcore) fp:", lib.fp(h1ao))
print("j1ao fp:", lib.fp(j1ao))
print("k1ao fp:", lib.fp(k1ao))

omega=0.0, hyb=0.25

WARN: MGGA Hessian is sensitive to dft grids. grids.level 3 may not be dense enough.



vxc_deriv1_ref fp: -3.4184689531771593


h1ao (hcore) fp: -34.59254245545264
j1ao fp: 35.78685953078744
k1ao fp: 1.5429165167985095


In [7]:
# 验证分解
f1ao_recap = vxc_deriv1_ref + h1ao + j1ao - 0.5 * hyb * k1ao
print("f1ao decomposition verified:", np.allclose(f1ao_recap, f1ao_ref))
print("max abs diff:", np.max(np.abs(f1ao_recap - f1ao_ref)))

f1ao decomposition verified: True
max abs diff: 1.3322676295501878e-15


## 3. 格点、AO、rho、vxc、fxc 准备

In [8]:
grids = dft.grid.Grids(mol)
grids.coords = dat0["grid_coords"]
grids.weights = weights = dat0["grid_weights"]
ngrids = len(weights)

ao = ni.eval_ao(mol, grids.coords, deriv=2)
print("ao shape:", ao.shape)

rho = ni.eval_rho2(mol, ao[:10], mo_coeff, mo_occ, None, "MGGA")
vxc, fxc = ni.eval_xc_eff(mf.xc, rho, 2, xctype="MGGA")[1:3]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

ao shape: (10, 43328, 49)
vxc shape: (5, 43328) fxc shape: (5, 5, 43328)


## 4. AO 导数指标常量

In [9]:
# 方向指标
TX, TY, TZ = 0, 1, 2
# AO 导数指标 (deriv=2 共有 10 个分量)
O = 0           # 值
X, Y, Z = 1, 2, 3          # 一阶导
XX, XY, XZ = 4, 5, 6       # 二阶导
YX, YY, YZ = 5, 7, 8       # (YX=XY)
ZX, ZY, ZZ = 6, 8, 9       # (ZX=XZ, ZY=YZ)

GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]

## 5. 自行实现 Vxc_deriv1

Vxc_deriv1 有两个部分：

### 5.1 ipip 部分：梯度级别的 Vxc 矩阵

与 `pyscf.grad.rks.get_vxc` 的计算完全相同。核心思想是计算一个完整 `(3, nao, nao)` 的矩阵 `v_ip`，然后在每个原子的 AO 切片上提取结果。

In [10]:
# wv = weights * vxc (带系数)
wv = weights * vxc     # [5, ngrids]
wv[0] *= 0.5           # LDA: 0.5 factor for symmetrization
wv[4] *= 0.5           # tau: 0.5 from tau = 1/2 sum_r |grad phi|^2

# --- _gga_grad_sum_ 第一部分：ao[t+1].T @ aow ---
# aow = wv[0]*ao[O] + wv[1]*ao[X] + wv[2]*ao[Y] + wv[3]*ao[Z]
aow = np.einsum("xg, xgu -> gu", wv[:4], ao[:4])  # [ngrids, nao]

vmat_ip = np.zeros((3, nao, nao))
for t in range(3):
    vmat_ip[t] += ao[t + 1].T @ aow

# --- _make_dR_dao_w：aow_d[t] = wv[0]*ao[t+1] + sum_r wv[r+1]*ao[GGA_CALLS[t][r]] ---
aow_d = np.array([wv[0][:, None] * ao[d] for d in [X, Y, Z]])
aow_d[TX] += wv[1][:, None] * ao[XX] + wv[2][:, None] * ao[XY] + wv[3][:, None] * ao[XZ]
aow_d[TY] += wv[1][:, None] * ao[YX] + wv[2][:, None] * ao[YY] + wv[3][:, None] * ao[YZ]
aow_d[TZ] += wv[1][:, None] * ao[ZX] + wv[2][:, None] * ao[ZY] + wv[3][:, None] * ao[ZZ]
for t in range(3):
    vmat_ip[t] += aow_d[t].T @ ao[O]

# --- _tau_grad_dot_：v_ip[t] += ao[GGA_CALLS[t][r]].T @ (wv[4] * ao[r+1]) ---
for r in range(3):
    aow_tau = wv[4][:, None] * ao[r + 1]  # [ngrids, nao]
    for t in range(3):
        vmat_ip[t] += ao[GGA_CALLS[t][r]].T @ aow_tau

# 验证：v_ip == -pyscf.grad.rks.get_vxc(ni, ...)
exc_ref, v_ip_ref = rks_grad.get_vxc(ni, mol, grids, mf.xc, dm0, max_memory=4000)
print("v_ip matches gradient Vxc matrix:", np.allclose(vmat_ip, -v_ip_ref, atol=1e-10))

v_ip matches gradient Vxc matrix: True


### 5.2 fxc 部分：密度格点响应 + fxc 核的反馈

fxc 部分的计算步骤：
1. 对每个原子 $A$，计算一阶密度格点 $\partial_{A_t} \xi_g^\chi$ (通过 `_make_dR_rho1`)
2. 与 fxc 核缩并：$\tilde{w}[t, \chi, g] = \sum_{\chi'} w_g f_g^{\chi\chi'} \cdot \mathrm{dR\_rho1}[t, \chi', g]$
3. 构造类 Vxc 矩阵：$(\tilde{V})_{\mu\nu}^{(t)} = \sum_{g\chi} \tilde{w}[t, \chi, g] \cdot \xi_{g\mu\nu}^\chi$
4. 与 ipip 部分叠加，取反对称并反号

In [11]:
# 准备 ao_dm0 用于 _make_dR_rho1
ao_dm0 = [numint._dot_ao_dm(mol, ao[i], dm0, None, (0, mol.nbas), mol.ao_loc_nr()) for i in range(4)]

# wf = weights * fxc  (加权二阶泛函导数)
wf = weights * fxc  # [5, 5, ngrids]

vmat_deriv1 = np.zeros((natm, 3, nao, nao))

for A in range(natm):
    # 5.2.1: dR_rho1[t, chi, g] = 一阶密度格点
    dR_rho1 = rks_hess._make_dR_rho1(ao, ao_dm0, A, aoslices, "MGGA")  # [3, 5, ngrids]
    
    # 5.2.2: wv_f[chi, t, g] = sum_{chi'} fxc[chi, chi', g] * dR_rho1[t, chi', g]
    # 注意：第一个 x 是 fxc 的行指标 (chi)，与 dR_rho1 的 chi 维度缩并
    wv_f = np.einsum("xyg, txg -> ytg", wf, dR_rho1)  # [5, 3, ngrids]
    wv_f[0] *= 0.5   # LDA: 0.5 for symmetrization
    wv_f[4] *= 0.25  # tau: extra 0.5 (0.5 * 0.25=0.125 total)
    
    # 5.2.3a: LDA+GGA 部分：vmat_fxc[t] += aow[t].T @ ao[0]
    # aow[t] = wv_f[0,t]*ao[0] + wv_f[1,t]*ao[1] + wv_f[2,t]*ao[2] + wv_f[3,t]*ao[3]
    aow_f = np.einsum("ctg, cgm -> tgm", wv_f[:4], ao[:4])  # [3, ngrids, nao]
    for t in range(3):
        vmat_deriv1[A, t] += aow_f[t].T @ ao[O]
    
    # 5.2.3b: tau 部分：vmat_fxc[t] += (wv_f[4,t] * ao[j]).T @ ao[j]  (j=1..3)
    for j in range(1, 4):
        for t in range(3):
            aow_tau = wv_f[4, t][:, None] * ao[j]
            vmat_deriv1[A, t] += aow_tau.T @ ao[j]
    
    # 5.3: 叠加 ipip 部分 (仅作用于原子 A 的行)
    _, _, p0, p1 = aoslices[A]
    vmat_deriv1[A, :, p0:p1, :] += vmat_ip[:, p0:p1, :]
    
    # 5.4: 反对称化并反号 (电子坐标 → 核坐标 convention)
    vmat_deriv1[A] = -vmat_deriv1[A] - vmat_deriv1[A].transpose(0, 2, 1)

In [12]:
# 验证与 PySCF 参考值一致
print("vmat_fxc (my) fp:", lib.fp(vmat_deriv1))
print("vxc_deriv1_ref fp:", lib.fp(vxc_deriv1_ref))
print("max abs diff:", np.max(np.abs(vmat_deriv1 - vxc_deriv1_ref)))
print("allclose:", np.allclose(vmat_deriv1, vxc_deriv1_ref, atol=1e-8))

vmat_fxc (my) fp: -3.418468953177159
vxc_deriv1_ref fp: -3.4184689531771593
max abs diff: 1.3322676295501878e-15
allclose: True


## 6. 组装 f1ao 并变换到 f1mo

In [13]:
# f1ao = h1ao + j1ao - hyb/2 * k1ao + vxc_deriv1
f1ao_my = h1ao + j1ao - 0.5 * hyb * k1ao + vmat_deriv1
print("f1ao_my vs ref:", np.allclose(f1ao_my, f1ao_ref))
print("max abs diff:", np.max(np.abs(f1ao_my - f1ao_ref)))

f1ao_my vs ref: True
max abs diff: 1.7763568394002505e-15


In [14]:
# f1mo = C^T @ f1ao @ C_occ
f1mo = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, f1ao_my, mocc)
print("f1mo shape:", f1mo.shape)
print("f1mo fp:  ", lib.fp(f1mo))

f1mo shape: (4, 3, 49, 5)
f1mo fp:   5.787136905641572


In [15]:
vmat_deriv1_mo = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, vmat_deriv1, mocc)
print("vmat_fxc_mo fp:", lib.fp(vmat_deriv1_mo))

vmat_fxc_mo fp: 0.5250822794774113


## 7. 验证：与 PySCF 直接所得 f1mo 比较

In [16]:
# 使用 PySCF 的 f1ao_ref 直接变换作为参考
f1mo_ref = np.einsum("up, Atuv, vi -> Atpi", mo_coeff, f1ao_ref, mocc)
print("f1mo_ref fp:", lib.fp(f1mo_ref))
print("f1mo matches ref:", np.allclose(f1mo, f1mo_ref))
print("max abs diff:", np.max(np.abs(f1mo - f1mo_ref)))

f1mo_ref fp: 5.787136905641581
f1mo matches ref: True
max abs diff: 9.020562075079397e-16


## 总结

成功实现了 RKS Hessian 中 f1mo 的 DFT xc 贡献部分的自行计算。关键点：

1. **Vxc_deriv1 = ipip 部分 + fxc 部分**，两者都在反对称化前叠加，然后通过 `-vmat - vmat.T` 转变为核坐标 convention
2. **ipip 部分**与梯度级别的 Vxc 矩阵完全相同，使用 `_gga_grad_sum_` + `_tau_grad_dot_` 的等效实现
3. **fxc 部分**通过 `_make_dR_rho1` 获得一阶密度格点变化，与 fxc 核缩并后构造类 Vxc 矩阵
4. **矩阵乘法中权重的放置**：tau 部分中权重放在 bra 边 (即 `(wv*ao[j]).T @ ao[j]` 而非 `ao[t+1].T @ (wv*ao[j])`)
5. J/K 部分直接调用 PySCF 的 DF routine (`_gen_jk`)，与 RHF 完全相同

In [17]:
dat = dict(np.load("nh3_r_tpss0_decomp.npz"))
dat.update({
    "vmat_ip": vmat_ip,
    "vxc_deriv1": vxc_deriv1_ref,
    "vmat_deriv1": vmat_deriv1,
    "vmat_deriv1_mo": vmat_deriv1_mo,
})
np.savez("nh3_r_tpss0_decomp.npz", **dat)